# 05. MHD Stability and Perturbed Equilibria

Session 04 measured perturbations: a frequency, a toroidal mode number modulo its alias step,
rational surfaces the mode could sit on. This session asks why such a perturbation can exist at all.

```text
Session 03   equilibrium + kinetic plasma state
      ↓
Session 04   measured perturbations and transient dynamics
      ↓
Session 05   stability + resistive layers + driven 3-D plasma response      <- here
      ↓
Session 06   many discharges: operational space and statistics
```

One linear operator runs through the whole session:

$$
\text{equilibrium} \rightarrow \text{linearization} \rightarrow \text{Fourier decomposition}
\rightarrow \text{energy principle}
\;\rightarrow\;
\begin{cases}
\delta W & \text{DCON: ideal stability} \\
q = m/n,\ \eta \neq 0 & \text{RDCON: tearing, } \Delta' \\
\mathbf f_{\mathrm{ext}} \neq 0 & \text{GPEC: driven 3-D perturbed equilibrium}
\end{cases}
$$

The design of the session follows issue [#1023](https://github.com/VEST-Tokamak/vaft/issues/1023).

## Session Overview

**The question.** Given an equilibrium and a measured or applied perturbation, is the plasma
intrinsically unstable, resistively unstable at a rational surface, or simply responding to an
external 3-D drive?

By the end of this session you will be able to:

- linearize ideal MHD about an axisymmetric equilibrium and say why the displacement
  $\boldsymbol\xi$ is the natural perturbation variable;
- explain why toroidal mode numbers $n$ separate in an axisymmetric equilibrium while the poloidal
  harmonics $m$ of one $n$ couple;
- read the energy principle: a displacement with $\delta W < 0$ means ideal instability;
- explain why ideal MHD cannot reconnect field lines, what a resistive layer is, and what
  $\Delta'$ measures;
- tell a homogeneous stability problem ($\mathcal L\boldsymbol\xi = 0$, DCON) from a driven one
  ($\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$, GPEC) — the same operator, two questions;
- describe VEST's three 3-D coil sets, build a sector-current pattern for a chosen $n$, and say which
  toroidal harmonics six sectors can and cannot make;
- use linearity to turn two runs into a whole relative-phase scan, and separate the model's
  excitation space from what the VEST power supplies can actually do.

**The equilibrium.** Discharge 39915, the discharge of sessions 02–04, through the packaged EFIT
g-file at 319 ms — the file the stability solvers are validated on. It is a *different
reconstruction* of the same instant from the one sessions 03 and 04 read out of the sample ODS, and
Step 1 puts the two side by side: the rational surfaces a stability code sees are only as good as
the reconstruction they come from. (DCON cannot yet map the sample ODS's reconstruction, #1246.)

**Two modes.** Everything in *offline* mode runs from packaged data: the equilibrium, its rational
surfaces, the 3-D coil geometry, the sector currents and their vacuum field. DCON, RDCON and GPEC are
external Fortran codes. They run in *lab* mode (`VAFT_TUTORIAL_MODE=lab` with `GPECHOME` set);
offline, their cells say what they would compute and skip. The lab path takes about eight minutes.

## Physical Context

### 1. From a measured perturbation to three possibilities

Session 04 ended with observables: $f$, $n$ (modulo the alias step), a $\delta B$ amplitude, SXR and
camera structure, a timing. They admit three broad explanations:

| explanation | what drives it | the question a code answers |
| --- | --- | --- |
| intrinsic **ideal** instability | pressure and current gradients, no reconnection | can a displacement lower the ideal-MHD energy? |
| **resistive tearing** instability | the same free energy, released by reconnection at $q = m/n$ | is the tearing index $\Delta'$ positive? |
| **driven 3-D response** | an external non-axisymmetric field (coils, error fields) | how does a stable plasma respond to a known drive? |

### 2. Linearization about an equilibrium

Session 03 wrote force balance, $\mathbf J_0\times\mathbf B_0 = \nabla p_0$. Perturb it,
$\mathbf B = \mathbf B_0 + \delta\mathbf B$, $\mathbf J = \mathbf J_0 + \delta\mathbf J$,
$p = p_0 + \delta p$, and describe the motion by the plasma **displacement** $\boldsymbol\xi(\mathbf x, t)$.
In ideal MHD the field is frozen into the moving fluid, so

$$
\delta\mathbf B = \nabla\times(\boldsymbol\xi\times\mathbf B_0),
\qquad
\rho_0\,\frac{\partial^2\boldsymbol\xi}{\partial t^2} = \mathcal F[\boldsymbol\xi],
$$

where $\mathcal F$ is the linear ideal-MHD force operator. It is built from the equilibrium alone:
the equilibrium decides which perturbations are possible.

### 3. Fourier decomposition

In an axisymmetric equilibrium, $\phi$ does not appear in $\mathcal F$, so each toroidal mode
number $n$ is a separate problem. The poloidal angle does appear (through $B_0(\theta)$, the shape),
so the poloidal harmonics of one $n$ **couple**:

$$
\boldsymbol\xi = \sum_n \Big[\sum_m \boldsymbol\xi_{m,n}(\psi)\, e^{i m\theta}\Big] e^{-i n\phi}\, e^{-i\omega t}.
$$

A tokamak eigenmode is therefore not one $(m, n)$ component. It is one $n$ with a spectrum of $m$ —
the "dominant $m$" DCON reports is the largest term, not the only one.

### 4. An eigenvalue problem

With $\boldsymbol\xi \propto e^{-i\omega t}$, $\;-\rho_0\omega^2\boldsymbol\xi = \mathcal F[\boldsymbol\xi]$.
$\mathcal F$ is self-adjoint, so $\omega^2$ is real:

- $\omega^2 > 0$ — the perturbation oscillates: stable;
- $\omega^2 < 0$ — it grows exponentially: unstable;
- $\omega^2 = 0$ — marginal stability.

### 5. The energy principle

$$
\delta W[\boldsymbol\xi] = -\frac12\int \boldsymbol\xi^*\cdot\mathcal F[\boldsymbol\xi]\,dV .
$$

If **any** admissible displacement has $\delta W < 0$, the equilibrium is ideally unstable; if
$\delta W > 0$ for all of them, it is stable. Force operator, eigenvalue problem and energy principle
are three representations of one linear problem. DCON solves it for each $n$ and reports the
energy eigenvalues: plasma ($W_p$), vacuum ($W_v$) and total ($W_t$).

Local limits of the same principle — the Mercier (interchange) criterion and high-$n$ ballooning —
are evaluated surface by surface rather than globally.

### 6. Why ideal MHD cannot tear

The ideal Ohm's law $\mathbf E + \mathbf v\times\mathbf B = 0$ freezes the field into the fluid:
field-line topology cannot change. At a rational surface, $q(\psi_s) = m/n$, a helical perturbation
with the same pitch as the field lines can only be resisted by bending them. Breaking and
reconnecting them — an island — needs something ideal MHD does not have.

### 7. Resistive layers and $\Delta'$

With resistivity, $\mathbf E + \mathbf v\times\mathbf B = \eta\mathbf J$. Away from the rational
surface $\eta$ is negligible and the ideal ("outer") solution holds; in a thin layer around it,
$\eta$ allows reconnection. The two solutions are **matched**. The outer solution's information is
one number per surface, the jump of its logarithmic derivative:

$$
\Delta' = \left[\frac{1}{\tilde\psi}\frac{d\tilde\psi}{dr}\right]_{r_s^-}^{r_s^+}.
$$

In the classical single-surface picture, $\Delta' > 0$ drives tearing. A toroidal plasma has many
rational surfaces per $n$ and they couple, so the real object is a **matrix** of $\Delta'$, and its
diagonal alone is not the whole story.

### 8. Stability and perturbed force balance share one operator

Linearize force balance directly: $\delta\mathbf F = \delta\mathbf J\times\mathbf B_0 +
\mathbf J_0\times\delta\mathbf B - \nabla\delta p = \mathcal F[\boldsymbol\xi]$. At marginal stability,
$\omega^2 = 0$, the dynamic equation becomes

$$
\boxed{\mathcal F[\boldsymbol\xi] = 0}
$$

— a static, linearly perturbed equilibrium. So the same operator answers two questions:

```text
             same linear ideal-MHD operator
                          |
          +---------------+----------------+
          |                                |
   homogeneous  L xi = 0             driven  L xi = f_ext
   does the equilibrium admit a      how does a stable plasma answer
   self-growing perturbation?        a known external 3-D field?
          |                                |
        DCON                             GPEC
```

A general dynamic eigenmode is not a static perturbed equilibrium; the identity is exact in the
marginal, $\omega \to 0$ limit, which is exactly where a slowly applied coil field lives.

### 9. GPEC: a linear 3-D perturbed equilibrium

GPEC keeps the axisymmetric equilibrium as the background and computes the linear response to a
small applied field, $\mathbf B = \mathbf B_0 + \delta\mathbf B_{3D}$ with $|\delta B| \ll |B_0|$. It
is not a nonlinear 3-D equilibrium solver. Its central decomposition is

$$
\delta\mathbf B_{\mathrm{total}} = \delta\mathbf B_{\mathrm{vacuum}} + \delta\mathbf B_{\mathrm{plasma}} .
$$

The field the coils apply is not the field inside the plasma: the plasma can **screen** a resonant
component (currents on the rational surface cancel it) or **amplify** a component that couples to a
weakly stable kink. Resonant and non-resonant parts both matter; GPEC is not a resonant-field
calculator.

### 10. Which code answers which question

| | DCON | RDCON | GPEC |
| --- | --- | --- | --- |
| model | ideal MHD | resistive matching across rational layers | ideal MHD with an external drive |
| equation | $\mathcal L\boldsymbol\xi = 0$ | outer ideal solution + inner layers | $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$ |
| key number | $\delta W$ (energy eigenvalues) | $\Delta'$ per surface, and its matrix | vacuum/plasma/total $\delta B_{m,n}$, resonant flux |
| structure | global eigenfunction $\xi_m(\psi)$ | rational surfaces and their coupling | 3-D field and displacement |
| typical physics | kink, interchange, ballooning | tearing | error-field and RMP response |

RDCON is not a nonlinear time-evolution code: it solves the matching problem. STRIDE solves the same
problem independently and is the natural cross-check of RDCON's $\Delta'$.

## Load / Prepare Data

### Imports and the execution mode

Offline is the default. Lab mode is asked for explicitly, and even then a solver runs only when its
executable is found; the preflight prints what will and will not run.

In [ ]:
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omas import ODS

import vaft
from vaft.code import gpec
from vaft.data.resources import data_path, sample_geqdsk
from vaft.formula import stability
from vaft.machine_mapping.coils_non_axisymmetric import (
    CoilExcitation, apply_coil_excitation, coils_non_axisymmetric,
)
from vaft.machine_mapping.coils_non_axisymmetric_geometry import load_vest_3d_coil_config
from vaft.process.coils_non_axisymmetric import biot_savart_filaments, toroidal_mode_decomposition
from vaft.process.magnetic_island import (
    as_equilibrium, equilibrium_safety_factor, find_rational_surfaces,
)
from vaft.process import perturbation

In [ ]:
MODE = os.environ.get("VAFT_TUTORIAL_MODE", "offline")
SOLVERS = {name: gpec.find_gpec_executable(name) for name in ("dcon", "rdcon", "gpec")}
RUN = {name: MODE == "lab" and path is not None for name, path in SOLVERS.items()}

print(f"mode: {MODE}")
for name, path in SOLVERS.items():
    state = "runs" if RUN[name] else ("skipped: offline mode" if MODE != "lab" else f"skipped: ${gpec.GPEC_HOME_ENV}/bin/{name} not found")
    print(f"  {name:5s} {state}")

### The equilibrium

One file feeds everything: the offline safety-factor profile below, and — in lab mode — the DCON,
RDCON and GPEC runs. The time is read from the file name, never typed. The sample ODS of sessions
03–04 is loaded beside it for the comparison in Step 1.

In [ ]:
SHOT = 39915
GEQDSK = data_path("efit/g039915.00319")
TIME_MS = int(GEQDSK.name.split(".")[-1])

equilibrium = as_equilibrium(GEQDSK)
eq_ods = sample_geqdsk(f"efit/{GEQDSK.name}").to_omas()
psi_n, q = equilibrium_safety_factor(equilibrium)
b0 = abs(float(eq_ods["equilibrium.vacuum_toroidal_field.b0"][0]))
r0 = float(eq_ods["equilibrium.vacuum_toroidal_field.r0"])
lcfs_r = np.asarray(eq_ods["equilibrium.time_slice.0.boundary.outline.r"], dtype=float)
lcfs_z = np.asarray(eq_ods["equilibrium.time_slice.0.boundary.outline.z"], dtype=float)
if np.hypot(lcfs_r[0] - lcfs_r[-1], lcfs_z[0] - lcfs_z[-1]) < 1e-9:
    lcfs_r, lcfs_z = lcfs_r[:-1], lcfs_z[:-1]   # a closed outline repeats its first point

reference = vaft.omas.sample_ods(SHOT)
reference_times = np.asarray(reference["equilibrium.time"], dtype=float)
i_same = int(np.argmin(np.abs(reference_times - TIME_MS * 1e-3)))

print(f"shot {SHOT} at {TIME_MS} ms, from {GEQDSK.name}")
print(f"q on axis {q[0]:.2f}, q at the boundary {q[-1]:.2f}")
print(f"B0 = {b0:.3f} T at R0 = {r0:.3f} m; LCFS R {lcfs_r.min():.3f}-{lcfs_r.max():.3f} m, "
      f"Z {lcfs_z.min():+.3f}-{lcfs_z.max():+.3f} m")

### The 3-D coils

The packaged VEST non-axisymmetric coil geometry: three sets of six sectors, each a single geometric
filament standing for a 20-turn winding. It is mapped into an ODS once, and every excitation in this
session is overlaid on that geometry.

In [ ]:
coil_config = load_vest_3d_coil_config()
coil_ods = ODS()
coils_non_axisymmetric(coil_ods)

print(f"{'set':4s} {'sectors':>7s} {'turns':>5s}  sector angles [deg]      description")
for name, coil_set in coil_config.coil_sets.items():
    angles = ", ".join(f"{a:g}" for a in coil_set.sector_angles_deg)
    print(f"{name:4s} {len(coil_set.filaments):7d} {coil_set.turns:5.0f}  {angles:24s} {coil_set.description}")

## Guided Analysis

### Step 1 — Where a perturbation can resonate

A helical perturbation $e^{i(m\theta - n\phi)}$ has the pitch of the field lines where
$q(\psi_s) = m/n$. There, field-line bending costs nothing: it is the place ideal MHD cannot hold
and resistive MHD can tear.

In [ ]:
vaft.diagram.rational_surface()

In [ ]:
surfaces = {n: find_rational_surfaces(psi_n, q, n) for n in (1, 2, 3)}

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(psi_n, q, color="k", label="q from the EFIT g-file")
for (n, found), marker in zip(surfaces.items(), ("o", "s", "^")):
    axes.plot(found["psi_n_rational"], found["q_rational"], marker, linestyle="none",
              label=f"n={n}: {found['m'].size} rational surfaces")
axes.set_xlabel(r"normalized poloidal flux $\psi_N$")
axes.set_ylabel("q")
axes.set_title(f"#{SHOT} at {TIME_MS} ms: where q = m/n")
axes.grid(alpha=0.4)
axes.legend()
plt.show()

n1 = surfaces[1]
print("n=1 rational surfaces:")
for m, where in zip(n1["m"], n1["psi_n_rational"]):
    print(f"  m={m:2d}  psi_N = {where:.3f}")
print(f"gap between the last five n=1 surfaces: {np.round(np.diff(n1['psi_n_rational'])[-4:], 3)}")

In [ ]:
ref_psi, ref_q = equilibrium_safety_factor(as_equilibrium(reference, time_index=i_same))
ref_surfaces = find_rational_surfaces(ref_psi, ref_q, 1)

figure, axes = plt.subplots(figsize=(8, 4.2))
axes.plot(psi_n, q, "k", label=f"{GEQDSK.name}: {n1['m'].size} n=1 surfaces")
axes.plot(ref_psi, ref_q, "C1--", label=f"sample ODS at {reference_times[i_same] * 1e3:.0f} ms: "
                                        f"{ref_surfaces['m'].size} n=1 surfaces")
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("q")
axes.set_title(f"Two EFIT reconstructions of #{SHOT} at the same time")
axes.grid(alpha=0.4)
axes.legend()
plt.show()
for m in (3, 4, 5):
    here = n1["psi_n_rational"][n1["m"] == m]
    there = ref_surfaces["psi_n_rational"][ref_surfaces["m"] == m]
    print(f"q = {m}: psi_N {here[0]:.3f} in the g-file, {there[0]:.3f} in the sample ODS"
          if here.size and there.size else f"q = {m}: missing from one reconstruction")

VEST's $q$ climbs from about 2 on axis to about 13 at the boundary, so a single $n = 1$ has eleven
rational surfaces, and they crowd together towards the edge: the last few are one or two percent of
$\psi_N$ apart. Two consequences follow for everything below:

- the $m = 2$ surface sits almost on the axis, where the reconstruction's core $q$ is least
  constrained — session 03 said as much, and an answer that depends on it inherits that uncertainty;
- closely spaced edge surfaces mean that islands there need only be small to **overlap** (Step 8).

The two reconstructions agree on the interior $q = 3, 4, 5$ surfaces to within a few percent of
$\psi_N$ but not on the edge: one reaches $q \approx 13$, the other $q \approx 8$, and the count of
$n = 1$ surfaces follows. Session 03's constraint weights and profile freedom are why. Every number
below belongs to *this* reconstruction.

### Step 2 — Ideal stability: DCON and $\delta W$

DCON solves the ideal-MHD energy principle for one $n$ at a time, with the vacuum outside the plasma
(free boundary). It reports three spectra of energy eigenvalues — plasma $W_p$, vacuum $W_v$, total
$W_t$ — and names the least-stable total one, `total1`, which decides the verdict: $W_t < 0$ means
some displacement lowers the energy.

The cost rises steeply with $n$ (about 10 s for $n = 1$ on this equilibrium, minutes for $n = 3$), so
the session scans $n = 1$ and leaves more modes to the exercise.

In [ ]:
DCON_MODES = (1,)
dcon = None
if RUN["dcon"]:
    # A short directory: the GPEC suite reads paths into fixed-length Fortran strings (#1239).
    dcon_root = Path(tempfile.mkdtemp(prefix="vaft-t05-dcon-"))
    started = time.monotonic()
    dcon = gpec.run_gpec_suite_case(
        gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=GEQDSK, workdir=dcon_root),
        gpec.GPECSuiteConfig(modules=("dcon",), modes=DCON_MODES),
    )
    print(f"DCON finished in {time.monotonic() - started:.0f} s")
    for record in dcon.records:
        print(f"  n={record.mode}: {record.status} (return code {record.returncode})")
else:
    print(f"DCON skipped: would solve delta-W for n = {DCON_MODES} on {GEQDSK.name}, about 10 s per n=1.")

In [ ]:
dcon_outputs = {}
if dcon is not None:
    for record in dcon.records:
        problems = gpec.validate_dcon_result(record.workdir, record.mode)
        if problems:
            print(f"n={record.mode}: unusable -- {'; '.join(problems)}")
            continue
        native = gpec.read_dcon_output(record.workdir, mode=record.mode)
        dcon_outputs[record.mode] = native
        total = np.asarray(native.W_t_eigenvalue).real
        least = native.total1.real   # selected by DCON's mode label, not by position
        verdict = "ideally STABLE" if least > 0 else "ideally UNSTABLE"
        print(f"n={record.mode}: least-stable W_t = {least:+.4g}  ->  {verdict}")
        print(f"      {total.size} eigenvalues, {int((total < 0).sum())} negative; "
              f"dominant poloidal harmonic m = {native.m_pol_dominant}; "
              f"limiting surface q = {float(native.qlim):.2f} at psi_N = {float(native.psilim):.3f}")
else:
    print("Would print the least-stable total energy eigenvalue for each n, how many are negative,\n"
          "and the dominant poloidal harmonic of that eigenvector.")

In [ ]:
if dcon_outputs:
    from vaft.omas.vest_upstream import build_mhd_linear_ods

    dcon_ods, _ = build_mhd_linear_ods(shot=SHOT, time_values=[TIME_MS], workdir=dcon_root,
                                       modules=("dcon",), modes=tuple(dcon_outputs))
    vaft.omas.plot_mhd_linear_overview_eigenfunction(dcon_ods, figsize=(13, 4.6))
    plt.show()
else:
    print("Would draw the least-stable eigenfunction: one displacement trace xi_m(psi) per poloidal\n"
          "harmonic, and the perturbed field DCON derives from it.")

Read the eigenfunction as a *spectrum*: one $n$, several $m$, each peaking near its own rational
surface — the coupling of Section 3 made visible. The verdict is one number; the structure is what
you compare with the $n$ that session 04 measured.

**Local criteria.** Global $\delta W$ is not the only ideal limit. The Mercier criterion tests
interchange surface by surface, and high-$n$ ballooning is tested on each field line; the
circular $s$–$\alpha$ model shows the idea — magnetic shear $s$ against the normalized pressure
gradient $\alpha$, with a first-stable, an unstable and a second-stable region.

In [ ]:
vaft.diagram.s_alpha_ballooning()

In [ ]:
for shear, alpha in ((1.0, 0.2), (1.0, 1.0), (0.2, 2.5)):
    stable = stability.s_alpha_ballooning_stable(shear, alpha)
    print(f"s = {shear:.1f}, alpha = {alpha:.1f}: {'stable' if stable else 'ballooning UNSTABLE'}")

The packaged `dcon.in` leaves ballooning off (`DCONOptions(bal_flag=False)`), and DCON writes a
criterion it never evaluated as exactly zero — which is also its marginal value. VAFT records what
was requested so that "marginal everywhere" is never confused with "not computed"; turning it on is
`gpec.GPECSuiteConfig(dcon=gpec.DCONOptions(bal_flag=True))`.

### Step 3 — Tearing: rational layers and $\Delta'$

Ideal MHD holds the topology fixed, so the rational surface is where resistivity matters. The
calculation splits the plasma into ideal outer regions and a thin resistive layer, and matches them.

In [ ]:
vaft.diagram.tearing_layer_matching()

In [ ]:
vaft.diagram.delta_prime()

$\Delta'$ is the jump in the outer solution's logarithmic slope. Two outer solutions with the same
value at the surface, one kinked inward (stable) and one outward:

In [ ]:
for label, inside, outside in (("kinked inward", 2.0, -3.0), ("kinked outward", -1.0, 1.5)):
    jump = stability.delta_prime_from_outer_derivatives(1.0, inside, outside)
    print(f"{label:15s} dpsi/dr = {inside:+.1f} inside, {outside:+.1f} outside  ->  "
          f"Delta' = {jump:+.1f} m^-1  ({'tearing drive' if jump > 0 else 'stable'})")

In [ ]:
RDCON_MODES = (1,)
rdcon_surfaces = {}
if RUN["rdcon"]:
    rdcon_root = Path(tempfile.mkdtemp(prefix="vaft-t05-rdcon-"))
    started = time.monotonic()
    rdcon = gpec.run_gpec_suite_case(
        gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=GEQDSK, workdir=rdcon_root),
        gpec.GPECSuiteConfig(modules=("rdcon",), modes=RDCON_MODES),
    )
    print(f"RDCON finished in {time.monotonic() - started:.0f} s")
    for record in rdcon.records:
        usable, why = gpec.SOLVERS["rdcon"].check_success(record.workdir, record.mode)
        if not usable:
            print(f"  n={record.mode}: unusable -- {why}")
            continue
        matching = gpec.read_pest3_matching_output(record.workdir, solver="rdcon", mode=record.mode)
        rdcon_surfaces[record.mode] = matching.delta_prime_diagonal()
else:
    print(f"RDCON skipped: would solve the rational-surface matching problem for n = {RDCON_MODES}, "
          "about a minute for n=1.")

In [ ]:
# Where a classical Delta' is worth reading on this equilibrium: away from the axis, where the core q
# is poorly constrained, and from the crowded edge surfaces, where the layers are not thin compared
# with their spacing. A judgement about this case, stated once and applied as a label, not a cut.
INTERIOR = (0.2, 0.9)

if rdcon_surfaces:
    for mode, rows in rdcon_surfaces.items():
        print(f"n={mode}: {len(rows)} rational surfaces")
        print(f"   {'m':>3} {'psi_N':>7} {'Re Delta-prime':>16}   reading")
        for row in rows:
            inside = INTERIOR[0] <= row["psi_n"] <= INTERIOR[1]
            reading = ("tearing UNSTABLE" if row["delta_prime_real"] > 0 else "tearing stable") if inside \
                else ("near the axis" if row["psi_n"] < INTERIOR[0] else "near the edge")
            print(f"   {row['m']:>3} {row['psi_n']:7.3f} {row['delta_prime_real']:16.4g}   {reading}")
    figure, axes = plt.subplots(figsize=(7, 4))
    for mode, rows in rdcon_surfaces.items():
        axes.plot([r["psi_n"] for r in rows], [r["delta_prime_real"] for r in rows], "o-", label=f"n={mode}")
    axes.set_yscale("symlog", linthresh=1.0)
    axes.axhline(0.0, color="0.5", linewidth=0.8)
    axes.axvspan(*INTERIOR, color="0.92", zorder=0, label="interpretable band")
    axes.set_xlabel(r"$\psi_N$ of the rational surface")
    axes.set_ylabel(r"Re $\Delta'$ (symlog)")
    axes.set_title("Classical tearing index per surface: above zero drives tearing")
    axes.legend()
    plt.show()
else:
    print("Would list every n=1 rational surface with its Delta', and draw Delta' against psi_N.")

In a lab run on this equilibrium the surfaces $m = 3, 4, 5$ come out with $\Delta' < 0$; from $m = 6$ at
$\psi_N \approx 0.87$ outward it is positive and grows by orders of magnitude towards the edge, and
the $m = 2$ surface on the axis carries a number many decades larger than anything else. Neither extreme is a physics result: near the axis it is the core-$q$
uncertainty of Step 1, near the edge the layers are not thin compared with the spacing between
surfaces. The labels, not a deleted row, carry that judgement.

**DCON and RDCON can disagree, and should be allowed to.** An equilibrium ideally stable at $n = 1$
can still have $\Delta' > 0$ somewhere: tearing uses the same free energy through reconnection,
which ideal MHD forbids. STRIDE solves the same matching problem independently and is the
cross-check for any $\Delta'$ you intend to quote.

### Step 4 — From stability to driven response: the VEST 3-D coils

Steps 2 and 3 asked the homogeneous question, $\mathcal L\boldsymbol\xi = 0$. The rest of the
session asks the driven one, $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$, with the drive
coming from VEST's non-axisymmetric coils. Session 04 drew the *diagnostic* geometry; this is the
*actuator* geometry.

In [ ]:
vaft.omas.plot_coil_3d_geometry_topview(coil_ods)
plt.show()

In [ ]:
vaft.omas.plot_coil_3d_geometry3d(coil_ods)
plt.show()

In [ ]:
figure, axes = plt.subplots(figsize=(4.8, 6.5))
axes.plot(eq_ods["wall.description_2d.0.limiter.unit.0.outline.r"],
          eq_ods["wall.description_2d.0.limiter.unit.0.outline.z"], color="0.4", label="limiter")
axes.plot(lcfs_r, lcfs_z, "k-", label="LCFS")
for name, coil_set in coil_config.coil_sets.items():
    points = np.concatenate([filament.points_xyz for filament in coil_set.filaments])
    axes.plot(np.hypot(points[:, 0], points[:, 1]), points[:, 2], ".", markersize=2, label=name)
axes.set_aspect("equal")
axes.set_xlabel("R [m]")
axes.set_ylabel("Z [m]")
axes.set_title("3-D coils projected on the poloidal plane")
axes.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.show()

UP and LOW are saddle arrays above and below the midplane — in the top view UP lies exactly over
LOW — and MID is a set of circular 12-inch coils on the outboard midplane, about 4 cm outside the
LCFS and by far the closest to the plasma. All three have six sectors, but UP and LOW sit at
$15^\circ, 75^\circ, \dots$ while MID sits at $0^\circ, 60^\circ, \dots$ — a sector's angle belongs to
its coil set, and a current pattern built on the wrong angles is a different pattern.

### Step 5 — A coil-current pattern is a toroidal harmonic

Give sector $k$ at angle $\phi_k$ the current

$$
I_k = I_0 \cos(n\phi_k + \phi_0),
$$

with $I_0$ the amplitude, $n$ the harmonic and $\phi_0$ its toroidal phase. The session uses
$I_0 = 200$ A per turn, the excitation of the 48226 reference run, on 20 turns.

In [ ]:
AMPLITUDE_A = 200.0   # per turn
N_TOR = 1

excitations = [
    CoilExcitation.from_mode(name, AMPLITUDE_A, N_TOR, 0.0, sector_angles_deg=coil_set.sector_angles_deg)
    for name, coil_set in coil_config.coil_sets.items()
]
apply_coil_excitation(coil_ods, excitations)
for excitation in excitations:
    print(f"{excitation.coil_set:4s} I_k = {np.round(excitation.currents_a, 1)} A")

vaft.omas.plot_coil_3d_profile_current(coil_ods)
plt.show()

In [ ]:
vaft.omas.plot_coil_3d_spectrum_current(coil_ods, modes=range(0, 8))
plt.show()

Six sectors resolve $|n| \le 2$; the title says so for everything above. Asking for $n = 5$ or 7
returns the $n = 1$ amplitude again, because on six points $\cos(5\phi_k)$, $\cos(7\phi_k)$ and
$\cos(\phi_k)$ are the *same numbers*. It is the actuator side of session 04's alias step:

```text
session 04: finite toroidal sensors   -> an observable n spectrum modulo the alias step
session 05: finite toroidal actuators -> a controllable n spectrum modulo the same step
```

The coils cannot make a pure $n = 1$ field. What they make is the next step's question.

### Step 6 — The vacuum field at the plasma boundary

The vacuum field of the excited filaments follows from Biot–Savart, with each sector current
multiplied by its 20 turns. Evaluate it on the LCFS at 48 toroidal angles, take the component
normal to the surface — the one that bends field lines out of it — and decompose it toroidally.

In [ ]:
phi_grid = np.linspace(0.0, 2.0 * np.pi, 48, endpoint=False)
tangent_r, tangent_z = np.gradient(lcfs_r), np.gradient(lcfs_z)
length = np.hypot(tangent_r, tangent_z)
normal_r, normal_z = tangent_z / length, -tangent_r / length
outward = np.sign(np.mean(normal_r * (lcfs_r - lcfs_r.mean()) + normal_z * (lcfs_z - lcfs_z.mean())))
normal_r, normal_z = outward * normal_r, outward * normal_z

R, PHI = np.meshgrid(lcfs_r, phi_grid, indexing="ij")
Z = np.broadcast_to(lcfs_z[:, None], R.shape)
probes = np.column_stack([(R * np.cos(PHI)).ravel(), (R * np.sin(PHI)).ravel(), Z.ravel()])

b_normal = {}
for excitation in excitations:
    coil_set = coil_config.coil_sets[excitation.coil_set]
    field = biot_savart_filaments([f.points_xyz for f in coil_set.filaments],
                                  np.asarray(excitation.currents_a) * coil_set.turns, probes).reshape(*R.shape, 3)
    b_r = field[..., 0] * np.cos(PHI) + field[..., 1] * np.sin(PHI)
    b_normal[excitation.coil_set] = b_r * normal_r[:, None] + field[..., 2] * normal_z[:, None]
print(f"{probes.shape[0]} points on the LCFS ({lcfs_r.size} poloidal x {phi_grid.size} toroidal)")

In [ ]:
modes = np.arange(0, 14)
figure, axes = plt.subplots(figsize=(8, 3.8))
for offset, (name, values) in zip((-0.2, 0.0, 0.2), b_normal.items()):
    strongest = int(np.argmax(np.abs(values).max(axis=1)))
    spectrum = toroidal_mode_decomposition(phi_grid, values[strongest], modes)
    # 2|C_n| is the peak amplitude of a cosine for n != 0; the mean (n = 0) is |C_0|.
    amplitude = np.array([(2.0 if n else 1.0) * abs(spectrum[n]) for n in modes]) * 1e4
    axes.plot(modes + offset, amplitude / amplitude[1], "o", label=f"{name}, at its strongest LCFS point")
    print(f"{name:4s} n=1 {amplitude[1]:6.2f} G   n=5/n=1 {amplitude[5] / amplitude[1]:.2f}   "
          f"n=7/n=1 {amplitude[7] / amplitude[1]:.2f}   n=2,3,4 < {amplitude[2:5].max():.1e} G")
axes.set_yscale("log")
axes.set_ylim(1e-4, 5.0)
axes.set_xticks(modes)
axes.set_xlabel("toroidal mode number n of the field")
axes.set_ylabel(r"$|b_{n}| / |b_{1}|$ normal to the LCFS")
axes.set_title("An n=1 current pattern drives n = 1, 5, 7, 11, 13 ...; the rest is rounding")
axes.legend(fontsize="small", loc="lower right")
plt.show()

The field is exactly as the sectors allow: $n = 1$ and its sidebands $n = 6k \pm 1$, and nothing at
$n = 2, 3, 4$ beyond rounding. How strong the sidebands are depends on distance. A high-$n$ field
decays faster away from the coil, so at the LCFS the UP and LOW sidebands are a tenth of $n = 1$ or
less. MID, 4 cm away, is different: its $n = 5$ and $n = 7$ are **as strong as $n = 1$**. Close to a
coil the field is a local bump, not a harmonic.

GPEC takes one $n$ per run, so an $n = 1$ run is the $n = 1$ part of this field and nothing else.
For MID that leaves out as much field at the boundary as it keeps. The sidebands resonate much
farther out, on $q = m/5$ and $m/7$ surfaces, but they are not zero.

In [ ]:
theta = np.degrees(np.arctan2(lcfs_z, lcfs_r - r0))
order = np.argsort(theta)
harmonic = {
    name: np.array([toroidal_mode_decomposition(phi_grid, row, [N_TOR])[N_TOR] for row in values])
    for name, values in b_normal.items()
}

figure, axes = plt.subplots(figsize=(8, 3.8))
for name, coefficient in harmonic.items():
    amplitude = 2.0 * np.abs(coefficient)
    axes.plot(theta[order], amplitude[order] * 1e4, label=name)
    print(f"{name:4s} max |b_n=1| = {amplitude.max() * 1e4:5.1f} G  ->  delta B / B0 = {amplitude.max() / b0:.2%}")
axes.set_xlabel(r"geometric poloidal angle on the LCFS $\theta$ [deg]")
axes.set_ylabel(r"$|b_{n=1}|$ [G]")
axes.set_title("Where each coil set's n=1 field lands on the plasma boundary")
axes.legend()
plt.show()

Each set lands where it sits: UP near $\theta \approx 90^\circ$, LOW near $-90^\circ$, MID on the
outboard midplane. The printed $\delta B/B_0$ is the check that Section 9's linear picture applies:
a percent-level perturbation is small; one of over ten percent — MID at five times this current —
would not be, and a linear response calculation would no longer be the right tool.

### Step 7 — Toroidal phase, relative phase and linearity

**One coil set.** Shifting $\phi_0$ rotates the whole pattern. An axisymmetric plasma cannot tell
one toroidal angle from another, so the response rotates with it and no amplitude changes:
$b_n(\phi_0) = b_n(0)\,e^{i\phi_0}$. Scanning the phase of a single set on its own is not a scan.

**Two coil sets.** The *relative* phase is physical. For UP and LOW driven at the same $n$,

$$
b(\Delta\phi) = b_{\mathrm{UP}} + b_{\mathrm{LOW}}\,e^{i\Delta\phi},
$$

and because the vacuum field — and GPEC's linear response — is linear in the currents,
$\mathbf y = \mathbf R\,\mathbf I$, two calculations give the whole scan. First check that claim
against a direct Biot–Savart evaluation at one phase.

In [ ]:
check_phase = np.pi / 2
low_set = coil_config.coil_sets["LOW"]
shifted = CoilExcitation.from_mode("LOW", AMPLITUDE_A, N_TOR, np.degrees(check_phase),
                                   sector_angles_deg=low_set.sector_angles_deg)
field = biot_savart_filaments([f.points_xyz for f in low_set.filaments],
                              np.asarray(shifted.currents_a) * low_set.turns, probes).reshape(*R.shape, 3)
direct = (field[..., 0] * np.cos(PHI) + field[..., 1] * np.sin(PHI)) * normal_r[:, None] + field[..., 2] * normal_z[:, None]
direct_harmonic = np.array([toroidal_mode_decomposition(phi_grid, row, [N_TOR])[N_TOR] for row in direct])
predicted = harmonic["LOW"] * np.exp(1j * check_phase)
print(f"LOW at phase {np.degrees(check_phase):.0f} deg: largest difference between prediction and direct "
      f"Biot-Savart = {np.abs(direct_harmonic - predicted).max() / np.abs(predicted).max():.1e} of the peak")

In [ ]:
relative = np.radians(np.arange(0.0, 360.0, 15.0))
top, midplane = int(np.argmax(lcfs_z)), int(np.argmax(lcfs_r))
rms, at_top, at_midplane = [], [], []
for delta in relative:
    combined = harmonic["UP"] + harmonic["LOW"] * np.exp(1j * delta)
    rms.append(np.sqrt(np.mean(np.abs(2.0 * combined) ** 2)))
    at_top.append(2.0 * abs(combined[top]))
    at_midplane.append(2.0 * abs(combined[midplane]))

figure, axes = plt.subplots(figsize=(8, 3.8))
axes.plot(np.degrees(relative), np.array(rms) * 1e4, "o-", label="RMS over the whole LCFS")
axes.plot(np.degrees(relative), np.array(at_top) * 1e4, "s--", label="top of the LCFS (UP's side)")
axes.plot(np.degrees(relative), np.array(at_midplane) * 1e4, "^-", label="outboard midplane (both sets)")
axes.set_xlabel(r"relative phase LOW - UP, $\Delta\phi$ [deg]")
axes.set_ylabel(r"$|b_{n=1}|$ [G]")
axes.set_title("Same n, same currents: the relative phase decides where the fields interfere")
axes.legend(fontsize="small")
plt.show()
for label, values in (("RMS over the LCFS", rms), ("top", at_top), ("outboard midplane", at_midplane)):
    values = np.array(values) * 1e4
    print(f"{label:18s} {values.min():5.2f} - {values.max():5.2f} G  (max at {np.degrees(relative[int(np.argmax(values))]):.0f} deg)")

In [ ]:
uniform = np.linspace(-180.0, 180.0, 256, endpoint=False)
poloidal_m = np.arange(-8, 9)
figure, axes = plt.subplots(figsize=(8, 3.8))
for delta, marker in ((0.0, "o"), (np.pi, "s")):
    combined = harmonic["UP"] + harmonic["LOW"] * np.exp(1j * delta)
    on_uniform = np.interp(uniform, theta[order], combined.real[order], period=360.0) \
        + 1j * np.interp(uniform, theta[order], combined.imag[order], period=360.0)
    spectrum = np.fft.fftshift(np.fft.fft(on_uniform)) / uniform.size
    m_axis = np.fft.fftshift(np.fft.fftfreq(uniform.size, d=1.0 / uniform.size)).astype(int)
    pick = np.isin(m_axis, poloidal_m)
    axes.plot(m_axis[pick], 2e4 * np.abs(spectrum[pick]), marker, label=rf"$\Delta\phi$ = {np.degrees(delta):.0f} deg")
axes.set_xticks(poloidal_m)
axes.set_xlabel("poloidal harmonic in the geometric angle (a proxy, not a flux-coordinate m)")
axes.set_ylabel("amplitude [G]")
axes.set_title("In phase and in antiphase, the same n lands in different poloidal harmonics")
axes.legend()
plt.show()

Each set dominates its own side of the boundary, so the top barely notices the relative phase and
neither does the RMS over the whole LCFS. Where the two fields meet — the outboard midplane — they
add at one relative phase and cancel at the opposite one.

The poloidal spectrum says what that means. The field sits in two lobes, near $\theta = +90^\circ$
(UP) and $-90^\circ$ (LOW), half a turn apart in $\theta$. With the lobes in phase the pattern
nearly repeats after half a poloidal turn, and that favours the even harmonics, $|m| = 2, 4, 6$;
with the lobes in antiphase it changes sign after half a turn, and the odd harmonics $|m| = 3, 5$
take over. (Midplane symmetry alone only decides cosine against sine; it is the placement of the two
lobes that selects even or odd $m$.) The same toroidal $n$, the same currents, a different
relative phase — and because each rational surface answers to its own $m = nq$, **the relative
phase chooses which surfaces are driven most**: $q = 2, 4$ in one case, $q = 3, 5$ in the other.

The spectrum here is taken in the geometric angle on the LCFS and in vacuum. A resonant
harmonic lives in a straight-field-line angle on each rational surface, and the plasma changes the
field it sees — both are what GPEC adds.

**Model space and hardware space are not the same.** VAFT and GPEC accept any six currents per set.
Whether VEST's power supplies can drive them — independent channels, amplitude and polarity limits,
phase control, timing, all three sets at once — is a separate, unverified question. Treat every
pattern in this session as a model excitation.

### Step 8 — The plasma's answer: GPEC

GPEC solves the driven problem on the same equilibrium, one coil set and one $n$ per run, after a
DCON run it reuses for the plasma's stability basis. Two runs — UP alone and LOW alone, both at
phase zero — are enough, by the linearity of Step 7, for the whole relative-phase scan.

Each case takes about three minutes. What they write:

- the **energies** of the perturbed state (vacuum, surface, plasma);
- the **field spectrum** $\delta B_{m,n}(\psi)$ in flux coordinates;
- the **resonant table**, surface by surface: the resonant flux and the island width it opens.

In [ ]:
GPEC_SETS = ("UP", "LOW")
gpec_roots = {}
if RUN["gpec"] and RUN["dcon"]:
    for name in GPEC_SETS:
        excitation = next(e for e in excitations if e.coil_set == name)
        root = Path(tempfile.mkdtemp(prefix=f"vaft-t05-{name}-"))
        config = gpec.GPECSuiteConfig(
            modules=("dcon", "gpec"), modes=(N_TOR,),
            gpec=gpec.IdealGPECOptions(coil_specs=(gpec.CoilInputSpec(name, tuple(excitation.currents_a)),)),
        )
        started = time.monotonic()
        result = gpec.run_gpec_suite_case(
            gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=GEQDSK, workdir=root), config)
        record = next(r for r in result.records if r.module == "gpec")
        print(f"{name}: GPEC {record.status} in {time.monotonic() - started:.0f} s")
        if record.status == "completed":
            gpec_roots[name] = root
else:
    print(f"GPEC skipped: would run DCON + GPEC for n={N_TOR} with {', '.join(GPEC_SETS)} driven alone, "
          "about three minutes each.")

In [ ]:
responses = {}
for name, root in gpec_roots.items():
    run = gpec.read_gpec_netcdf(root / f"{TIME_MS:05d}" / "gpec" / f"nn={N_TOR}", mode=N_TOR)
    responses[name] = run
    control, profile = run.control, run.profile
    print(f"{name}: energies  vacuum {control.energy_vacuum:.3g}  surface {control.energy_surface:.3g}  "
          f"plasma {control.energy_plasma:.3g} J")
    print(f"      {profile.psi_n_rational.size} rational surfaces;  vacuum resonant field "
          f"{np.abs(profile.Phi_res_v).min() * 1e4:.2f}-{np.abs(profile.Phi_res_v).max() * 1e4:.2f} G;  "
          f"total (with plasma response): {'present' if profile.Phi_res is not None else 'not written'}")
if not gpec_roots:
    print("Would print, per coil set, the energies of the perturbed state and the resonant field on\n"
          "each n=1 rational surface.")

In [ ]:
if "UP" in gpec_roots:
    from vaft.omas.vest_upstream import build_gpec_ideal_ods

    gpec_ods, _ = build_gpec_ideal_ods(shot=SHOT, time_values=[TIME_MS], workdir=gpec_roots["UP"], modes=(N_TOR,))
    vaft.omas.plot_mhd_linear_field_spectrum(gpec_ods)
    plt.show()
else:
    print("Would draw the n=1 field spectrum delta B_{m,n}(psi) the UP set drives, in flux coordinates.")

**What the packaged setup can and cannot give yet.** With the packaged DCON namelist, GPEC writes
the *vacuum* resonant field on every surface but not the *total* one with the plasma response: the
namelist's `con_flag` switches GPEC's singular-field output off, and the other setting stops DCON at
the first surface on this high-$q$ equilibrium (#1243). So the scan below is the
**vacuum-approximation** answer, and it is labelled as one. It is still the right place to start:
the vacuum island picture is what the plasma response is compared against — screening shrinks it,
amplification grows it.

In [ ]:
if set(GPEC_SETS) <= set(responses):
    up, low = responses["UP"].profile, responses["LOW"].profile
    chi1 = (responses["UP"].control.attrs or {}).get("chi1")
    if chi1 is None or not np.isfinite(chi1):
        raise RuntimeError("the GPEC control output carries no chi1: island widths cannot be scaled")
    chi1 = float(chi1)
    window = perturbation.ResonantWindow("interior", *INTERIOR, source="Step 3's interpretable band")
    scan_rms, scan_overlap = [], []
    for delta in relative:
        flux = up.Phi_res_v + low.Phi_res_v * np.exp(1j * delta)
        width = stability.island_width_from_resonant_flux(flux, up.area_rational, up.q_rational,
                                                          up.dqdpsi_n_rational, up.m_rational, chi1)
        scan_rms.append(perturbation.rms_resonant_field(up.psi_n_rational, flux, window=window))
        scan_overlap.append(perturbation.chirikov(up.psi_n_rational, width).max())
    figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.plot(np.degrees(relative), np.array(scan_rms) * 1e4, "o-")
    left.set_ylabel("RMS vacuum resonant field, interior band [G]")
    right.plot(np.degrees(relative), scan_overlap, "s-")
    right.axhline(1.0, color="r", linestyle="--", label="islands touch")
    right.set_ylabel("largest vacuum Chirikov parameter")
    right.legend()
    for axis in (left, right):
        axis.set_xlabel(r"relative phase LOW - UP, $\Delta\phi$ [deg]")
    figure.suptitle("Vacuum approximation: no screening, no amplification")
    plt.show()
    print(f"interior resonant field is largest at {np.degrees(relative[int(np.argmax(scan_rms))]):.0f} deg "
          f"and smallest at {np.degrees(relative[int(np.argmin(scan_rms))]):.0f} deg")
else:
    print("Would scan the relative phase with the two GPEC runs, as Step 7 did with the vacuum field:\n"
          "RMS resonant field over the interior band, and the largest island-overlap (Chirikov) parameter.")

Two lessons, whichever numbers your run prints:

- **The relative phase moves the resonant drive**, surface by surface — exactly the effect of
  Step 7, now on flux-coordinate rational surfaces.
- **The vacuum islands of VEST's crowded edge overlap easily.** A Chirikov parameter above one on
  the outer surfaces is the vacuum approximation saying "stochastic edge". Whether the plasma
  really lets that field in is precisely what the total response decides — the reason GPEC exists.

**From scan to optimization.** Write the response as $\mathbf y = \mathbf R\mathbf I$ with
$\mathbf I$ the coil currents and $\mathbf R$ an equilibrium-dependent response matrix. A phase scan
samples one circle of $\mathbf I$; an optimization asks for
$\mathbf I^* = \arg\min_{\mathbf I} J(\mathbf R\mathbf I)$ — minimize the resonant error field
$\sum_m |b^{\mathrm{res}}_{m,n}|^2$, maximize one surface or the edge, subject to current limits.
`vaft.process.perturbation.dominant_mode` gives the leading direction of a coupling matrix;
the full coil phase-space calculation is #1178.

## Interpretation Checkpoints

Each has a definite answer in what you have computed or read.

1. **DCON reports the least-stable $W_t > 0$ at $n = 1$.** Does that make the equilibrium stable to
   tearing? Which step answers that, and why can the two answers differ?
2. **The $m = 2$ surface's $\Delta'$ is many decades larger than any other.** Why is that not a
   tearing verdict, and which uncertainty from session 03 does it inherit?
3. **The dominant harmonic of the $n = 1$ eigenfunction is one $m$.** Is the eigenmode a single
   $(m, n)$ component? What did Section 3 say, and what does the eigenfunction plot show?
4. **An $n = 1$ sector pattern drives $n = 5$ and $n = 7$ fields.** Why, and which coil set keeps
   the most of them? What is the corresponding statement in session 04?
5. **Shifting the phase of the MID set alone changes nothing GPEC reports in amplitude.** Why is
   that not a bug, and what kind of phase scan *is* physical?
6. **Two GPEC runs give a 24-point phase scan.** What assumption allows it, and what quantity
   printed in Step 6 tells you the assumption is reasonable here?
7. **The vacuum Chirikov parameter exceeds one on the outer surfaces.** Does that prove a
   stochastic edge? What is missing from the calculation (#1243)?
8. **A pattern with arbitrary currents on all 18 sectors gives the smallest resonant field.** What
   must you check before calling it a VEST operating scenario?

## Integrated Analysis

### Part II, level 1 — two times, two reconstructions

Session 04 followed the rational surfaces through the sample ODS over time. A stability answer
depends on the reconstruction *and* the time, and both move it. The packaged legacy g-files cover
two times of 39915; lay them beside the sample ODS at the same times, then — in lab mode — ask DCON
the $n = 1$ question on each legacy file.

In [ ]:
legacy = sorted(data_path("efit").glob(f"g0{SHOT}.*"))
figure, axes = plt.subplots(figsize=(8, 4.2))
for path, colour in zip(legacy, ("C0", "C3")):
    when = int(path.name.split(".")[-1])
    slice_psi, slice_q = equilibrium_safety_factor(as_equilibrium(path))
    index = int(np.argmin(np.abs(reference_times - when * 1e-3)))
    other_psi, other_q = equilibrium_safety_factor(as_equilibrium(reference, time_index=index))
    axes.plot(slice_psi, slice_q, color=colour, label=f"{path.name}")
    axes.plot(other_psi, other_q, color=colour, linestyle="--",
              label=f"sample ODS at {reference_times[index] * 1e3:.0f} ms")
    found = find_rational_surfaces(slice_psi, slice_q, 1)
    q3 = found["psi_n_rational"][found["m"] == 3]
    print(f"{path.name}: q0 = {slice_q[0]:.2f}, n=1 surfaces m = {found['m'].min()}..{found['m'].max()}, "
          + (f"q = 3 at psi_N = {q3[0]:.3f}" if q3.size else "no q = 3 surface at all"))
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("q")
axes.legend(fontsize="small")
axes.grid(alpha=0.4)
plt.show()

Two milliseconds apart, the legacy reconstructions disagree about whether the core has a $q = 3$
surface at all: $q_0$ jumps from about 3.5 to about 2. Nothing in the discharge does that in 2 ms;
the core $q$ of a magnetics-only reconstruction is weakly constrained, and it moves with the fit.
The edge surfaces, pinned by the boundary, barely move. A stability result on the core modes —
the $m = 2$ and $m = 3$ surfaces, the least-stable eigenfunction's dominant harmonic — is only as
firm as that core $q$.

In [ ]:
if RUN["dcon"]:
    for path in legacy:
        when = int(path.name.split(".")[-1])
        root = Path(tempfile.mkdtemp(prefix="vaft-t05-time-"))
        record = gpec.run_gpec_suite_case(
            gpec.GPECCaseInputs(shot=SHOT, time_ms=when, geqdsk=path, workdir=root),
            gpec.GPECSuiteConfig(modules=("dcon",), modes=(1,)),
        ).records[0]
        if gpec.validate_dcon_result(record.workdir, 1):
            print(f"{path.name}: DCON output unusable")
            continue
        native = gpec.read_dcon_output(record.workdir, mode=1)
        print(f"{path.name}: least-stable W_t = {native.total1.real:+.4g}, "
              f"dominant m = {native.m_pol_dominant}")
else:
    print("Would run DCON n=1 on each legacy g-file and print the least-stable W_t per time.")

### Part II, level 2 — more than one $n$

The same six sectors make $n = 2$ and $n = 3$ patterns too. $n = 2$ is still resolved; $n = 3$ is
the Nyquist harmonic of six sectors, where a cosine pattern alternates in sign and its phase is no
longer free.

In [ ]:
for n in (2, 3):
    pattern_ods = ODS()
    coils_non_axisymmetric(pattern_ods)
    apply_coil_excitation(pattern_ods, [
        CoilExcitation.from_mode(name, AMPLITUDE_A, n, 30.0, sector_angles_deg=coil_set.sector_angles_deg)
        for name, coil_set in coil_config.coil_sets.items()
    ])
    vaft.omas.plot_coil_3d_spectrum_current(pattern_ods, modes=range(0, 7))
    plt.show()
    print(f"n={n} rational surfaces: m = {surfaces[n]['m'].min()}..{surfaces[n]['m'].max()} "
          f"({surfaces[n]['m'].size} surfaces)" if n in surfaces else "")

### Part II, level 3 — back to the measurement

| measured in session 04 | computed in session 05 |
| --- | --- |
| frequency $f$, laboratory frame | nothing here: the calculations are static ($\omega \to 0$) |
| $n$ modulo the alias step | one $n$ per DCON/RDCON/GPEC run; the coils' own alias step |
| candidate rational surfaces for that $n$ | $q = m/n$ surfaces and their positions over time |
| $\delta B$ at the probes, SXR and camera structure | eigenfunction $\xi_{m,n}(\psi)$, $\delta B_{m,n}$, island widths |
| a transient as a sequence of numbers | $\delta W$ and $\Delta'$ on the slice before it |

The chain closes only when the model's quantity is taken to the diagnostic: a synthetic Mirnov
signal from the eigenfunction or from GPEC's field at the probe positions. That comparison, not a
verdict on its own, is what identifies a measured mode.

## Independent Exercise

### A stability and response study of one equilibrium

Take the 39915 equilibrium (offline) or a reconstructed equilibrium of your own discharge (lab mode)
and write a short report:

1. **Validity of the equilibrium**: its time, $q$ on axis and at the edge, and how well the core $q$
   is constrained (session 03).
2. **Rational surfaces** for $n = 1$ and $n = 2$, and their spacing near the edge.
3. **DCON** for $n = 1$ and $n = 2$: the least-stable $W_t$, the number of negative eigenvalues,
   the dominant $m$, and the eigenfunction's harmonic content.
4. **RDCON** $\Delta'$ per surface, with the interpretable band you choose and why.
5. **Why DCON and RDCON may differ** on your case.
6. **A coil pattern** for a chosen coil set and $n$; its sector currents and current spectrum.
7. **The vacuum field** of that pattern on the LCFS: its toroidal sidebands and $\delta B/B_0$.
8. **A relative-phase scan** between two sets, first in vacuum (offline), then with GPEC (lab).
9. **Which phase maximizes or minimizes** your chosen metric, and whether the answer changes between
   the vacuum and the GPEC calculation.
10. **Hardware feasibility**: state separately whether the excitation is known to be achievable.
11. **Comparison with session 04**: which measured $n$ and which surfaces your results speak to,
    and what synthetic signal would make the comparison quantitative.

In [ ]:
# Lab mode: VAFT_TUTORIAL_MODE=lab and $GPECHOME. Replace the equilibrium, then rerun from the top.
# GEQDSK = Path("<your g-file>")
#
# 3. a DCON n-scan (cost rises steeply with n)
# DCON_MODES = (1, 2)
#
# Ballooning on the DCON run:
# config = gpec.GPECSuiteConfig(modules=("dcon",), modes=DCON_MODES, dcon=gpec.DCONOptions(bal_flag=True))
#
# 6. another pattern: the MID set alone at n=2 and 45 degrees
# coil_set = coil_config.coil_sets["MID"]
# pattern = CoilExcitation.from_mode("MID", AMPLITUDE_A, 2, 45.0, sector_angles_deg=coil_set.sector_angles_deg)
# apply_coil_excitation(coil_ods, [pattern])
# vaft.omas.plot_coil_3d_spectrum_current(coil_ods)
#
# 8. the relative phase of MID against UP: add "MID" to GPEC_SETS and combine
# flux = responses["UP"].profile.Phi_res_v + responses["MID"].profile.Phi_res_v * np.exp(1j * delta)

## Takeaways and Next Steps

- Ideal stability, tearing and driven 3-D response are **one linear operator asked three
  questions**: $\mathcal L\boldsymbol\xi = 0$ with $\delta W$ (DCON), the same outer solution matched
  across resistive layers (RDCON, $\Delta'$), and $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$
  (GPEC). Marginal stability *is* a static perturbed equilibrium.
- An eigenmode is one $n$ with a spectrum of coupled $m$; a verdict is one number, the structure is
  what meets the measurement.
- DCON and RDCON can disagree, and a $\Delta'$ near the axis or the crowded edge is a resolution
  statement before it is a physics one.
- Six sectors make $n$ modulo six, in the currents and in the field — the actuator-side alias step.
- A single set's toroidal phase rotates the answer; the **relative** phase between sets changes it.
  Linearity turns two runs into a whole scan.
- The vacuum field is not the field in the plasma. The packaged chain gives the vacuum resonant
  field today; the total response is #1243, and an offline stability sample is #1231.
- The model's excitation space is not VEST's hardware space.

**Method depth**: `notebooks/linear_ideal_stability_analysis_with_dcon.ipynb`,
`notebooks/linear_resistive_stability_analysis_with_rdcon.ipynb`, and the GPEC reference notebook
(#1237). Open work: DCON closure and Mercier/ballooning (#940, #527), RDCON IMAS mapping (#939), coil
phase-space optimization (#1178), 3-D harmonic diagrams (#1088).

**Next**: Session 06 leaves the single discharge. It asks which states the machine actually reaches
across many shots — the operational space — and where the limits of this session sit in it.